In [13]:
import cv2
import numpy as np
import os
import mediapipe as mp

# MediaPipe setup
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_face_mesh = mp.solutions.face_mesh_connections
mp_pose = mp.solutions.pose_connections
mp_hands = mp.solutions.hands_connections

# Helper functions
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_face_mesh.FACEMESH_CONTOURS,
                            mp_drawing_styles.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
                            mp_drawing_styles.DrawingSpec(color=(80,255,121), thickness=1, circle_radius=1))
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                            mp_drawing_styles.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                            mp_drawing_styles.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2))
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_hands.HAND_CONNECTIONS,
                            mp_drawing_styles.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                            mp_drawing_styles.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2))
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_hands.HAND_CONNECTIONS,
                            mp_drawing_styles.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                            mp_drawing_styles.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))

def extract_keypoints(results):
    # Only pose + hands (no face) — reduces noise significantly
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, lh, rh])  # 258 features instead of 1662

# Data config
DATA_PATH = os.path.join('LSTM_DATA')
actions = np.array(['hello', 'thanks', 'iloveyou'])
no_sequences = 30
sequence_length = 30
NUM_FEATURES = 258  # pose(132) + lh(63) + rh(63)

In [14]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

label_map = {label: num for num, label in enumerate(actions)}

sequences, labels = [], []
for action in actions:
    for sequence in range(no_sequences):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), str(frame_num) + '.npy'))
            # Slice out face landmarks: keep pose(0:132) + lh(1536:1599) + rh(1599:1662)
            res_no_face = np.concatenate([res[:132], res[1536:]])
            window.append(res_no_face)
        sequences.append(window)
        labels.append(label_map[action])

# Data augmentation — multiply dataset 3x
def augment_sequence(seq):
    seq = np.array(seq)
    augmented = []
    # Gaussian noise
    noisy = seq + np.random.normal(0, 0.01, seq.shape)
    augmented.append(noisy)
    # Time shift
    shift = np.random.randint(1, 4)
    shifted = np.roll(seq, shift, axis=0)
    augmented.append(shifted)
    return augmented

aug_sequences, aug_labels = [], []
for i in range(len(sequences)):
    aug_sequences.append(sequences[i])
    aug_labels.append(labels[i])
    for aug in augment_sequence(sequences[i]):
        aug_sequences.append(aug.tolist())
        aug_labels.append(labels[i])

X = np.array(aug_sequences, dtype=np.float32)
y = to_categorical(aug_labels)

print(f"Dataset: {X.shape[0]} samples, shape={X.shape}")
print(f"Labels: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Dataset: 270 samples, shape=(270, 30, 258)
Labels: (270, 3)


In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

model = Sequential([
    LSTM(64, return_sequences=True, activation='relu', input_shape=(30, NUM_FEATURES)),
    BatchNormalization(),
    Dropout(0.3),
    LSTM(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(actions.shape[0], activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_3 (LSTM)               (None, 30, 64)            82688     
                                                                 
 batch_normalization (BatchN  (None, 30, 64)           256       
 ormalization)                                                   
                                                                 
 dropout_3 (Dropout)         (None, 30, 64)            0         
                                                                 
 lstm_4 (LSTM)               (None, 128)               98816     
                                                                 
 batch_normalization_1 (Batc  (None, 128)              512       
 hNormalization)                                                 
                                                                 
 dropout_4 (Dropout)         (None, 128)              

In [16]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=16,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/150
14/14 [==============================] - 4s 60ms/step - loss: 0.9465 - accuracy: 0.6157 - val_loss: 0.9740 - val_accuracy: 0.5741 - lr: 5.0000e-04
Epoch 2/150
14/14 [==============================] - 0s 31ms/step - loss: 0.3470 - accuracy: 0.8843 - val_loss: 0.9311 - val_accuracy: 0.6296 - lr: 5.0000e-04
Epoch 3/150
14/14 [==============================] - 0s 32ms/step - loss: 0.2712 - accuracy: 0.8843 - val_loss: 0.8727 - val_accuracy: 0.7963 - lr: 5.0000e-04
Epoch 4/150
14/14 [==============================] - 0s 31ms/step - loss: 0.1828 - accuracy: 0.9537 - val_loss: 0.8137 - val_accuracy: 0.7778 - lr: 5.0000e-04
Epoch 5/150
14/14 [==============================] - 0s 32ms/step - loss: 0.1589 - accuracy: 0.9398 - val_loss: 0.7649 - val_accuracy: 0.9074 - lr: 5.0000e-04
Epoch 6/150
14/14 [==============================] - 0s 31ms/step - loss: 0.2027 - accuracy: 0.9398 - val_loss: 0.7290 - val_accuracy: 0.9074 - lr: 5.0000e-04
Epoch 7/150
14/14 [===========================

In [17]:
from sklearn.metrics import classification_report, confusion_matrix

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}\n")

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)
print(classification_report(y_true, y_pred, target_names=actions))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 12ms/step - loss: 0.0620 - accuracy: 1.0000
Test Accuracy: 100.00%
Test Loss: 0.0620

              precision    recall  f1-score   support

       hello       1.00      1.00      1.00        18
      thanks       1.00      1.00      1.00        18
    iloveyou       1.00      1.00      1.00        18

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[18  0  0]
 [ 0 18  0]
 [ 0  0 18]]


In [18]:
os.makedirs('model', exist_ok=True)
model.save(os.path.join('model', 'lstm_model.h5'))
print("Model saved!")

Model saved!


In [19]:
from tensorflow.keras.models import load_model

model = load_model(os.path.join('model', 'lstm_model.h5'))
colors = [(245,117,16), (117,245,16), (16,117,245)]

sequence = []
sentence = []
threshold = 0.6

cap = cv2.VideoCapture(0)
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        image, results = mediapipe_detection(frame, holistic)
        draw_styled_landmarks(image, results)

        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
            predicted_action = actions[np.argmax(res)]
            confidence = np.max(res)

            if confidence > threshold:
                if len(sentence) == 0 or predicted_action != sentence[-1]:
                    sentence.append(predicted_action)
                if len(sentence) > 3:
                    sentence = sentence[-3:]

                # Show prediction
                color = colors[np.argmax(res) % len(colors)]
                cv2.putText(image, f'{predicted_action} ({confidence*100:.1f}%)', (15, 40),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2, cv2.LINE_AA)

            # Show sentence history
            cv2.putText(image, ' '.join(sentence), (15, 80),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

        cv2.imshow('Ishara - LSTM Real-time Detection', image)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

KeyboardInterrupt: 

In [20]:
cv2.destroyAllWindows()
cv2.waitKey(1)

# Release any lingering captures
try:
    cap.release()
except:
    pass